# Širenje poremećaja u mreži aerodroma
Primjena Sorensen–Zhou / Hammarlingova algoritma na dataset
[OPDI v0.0.2 – europski letovi, EUROCONTROL PRC / OpenSky](https://www.opdi.aero/flight-list-data).
Mrežu čine opaženi letovi između europskih aerodroma u siječnju 2025.

**Pitanje:** kako mjesto nastanka poremećaja i brzina oporavka utječu na
varijabilnost cijele mreže? Stvarni letovi određuju veze, a dinamiku zadajemo
pojednostavljenim difuzijskim modelom. Ne procjenjujemo niti prognoziramo stvarna kašnjenja.

Pokrenite ćelije redom u kernelu **Julia Hammarling** iz ovog projekta.
Izvor bilježnice je `flight_delays_lyapunov.jl`; pomoćne funkcije su u `flight_network.jl`.


In [1]:
using LinearAlgebra
using BenchmarkTools
import MatrixEquations

project_root = isfile(joinpath(pwd(), "flight_network.jl")) ? pwd() : @__DIR__
include(joinpath(project_root, "flight_network.jl"))
using .FlightNetwork

module AirportLyapunov
    include(joinpath(@__DIR__, "sorensen_zhou.jl"))
end
using .AirportLyapunov: lyapunov_factor, relative_residual


(real_residual = 4.678790580762225e-17, complex_residual = 6.587146713995547e-17, relative_solution_error = 1.1101076597580674e-16)

Test Summary: | Pass  

Total  Time
Sorensen–Zhou |  134    134  9.3s


## 1. Podaci i parametri
`download_data = true` preuzima samo odabrani mjesečni OPDI Parquet
(`flight_list_202501.parquet`, oko 34 MB) i `airports.csv` iz OurAirports repozitorija
u `data/europe-flights/`. Postojeće datoteke koriste se ponovno; podaci se ne dodaju u Git.
OurAirports snapshot zaključan je na commit `7dd521a50a421171db1cb69f87657a60b48561c7`.
Za ručno preuzimanje koristite [OPDI portal](https://www.opdi.aero/flight-list-data)
i [OurAirports CSV](https://raw.githubusercontent.com/davidmegginson/ourairports-data/7dd521a50a421171db1cb69f87657a60b48561c7/airports.csv),
a zatim postavite `download_data = false`. Promjena godine/mjeseca odabire drugu datoteku.

OPDI daje pojedinačne opažene letove potrebne za veze mreže. Službene agregacije
EUROCONTROL-a i UK CAA ne daju te veze na razini pojedinačnih letova.
[Hugging Face izvedenica](https://huggingface.co/datasets/345rf4gt56t4r3e3/flight-delays-europe-2023-2025)
računa odstupanje trajanja od medijana rute, a ne kašnjenje prema redu letenja;
njezina kartica upozorava da nije točan istraživački izvor. Zato koristimo izvorni OPDI.

Početna ćelija učitava izvornu implementaciju te pokreće njezin primjer i provjere.
Ovdje ne kopiramo algoritam, pa promjene u `sorensen_zhou.jl` vrijede i za ovu analizu.


In [2]:
data_directory = get(ENV, "FLIGHT_DELAYS_DATA", joinpath(project_root, "data", "europe-flights"))
download_data = true
year = 2025
month = 1
number_of_airports = 50
alpha = 1.0
gamma = 0.2
sigma = 1.0
background_noise = 0.01

data_paths = ensure_dataset(data_directory; year, month, download=download_data)


(flights = "/home/carjin/projects/hammarlingov-algoritam/data/europe-flights/flight_list_202501.parquet", airports = "/home/carjin/projects/hammarlingov-algoritam/data/europe-flights/airports.csv")

## 2. Čišćenje i agregiranje
Iz Parqueta učitavamo samo `dof`, `adep` i `ades`, po grupama redaka, te provjeravamo
godinu i mjesec. Koristimo opažene ICAO oznake `adep`/`ades`, bez nadopune predviđenim
oznakama `adep_p`/`ades_p`. Izbacujemo nepoznata/nedostajuća polazišta ili odredišta
i letove koji se vraćaju u isto polazište.
Oba aerodroma moraju imati valjane koordinate i ICAO oznaku (`icao_code`) u OurAirports,
`continent = EU` te ne smiju imati `type = closed`. To je geografska Europa prema
tom izvoru, a ne Europska unija ni cijelo područje EUROCONTROL-a: npr. UK i Švicarska
su uključeni, Cipar i azijski dio Turske nisu. Odbačeni redci uključuju i veze izvan tog područja.

OPDI nema oznaku otkazanog leta: odsutnost opažanja ne tumačimo kao otkazivanje.
Uzorak ovisi o ADS-B pokrivenosti i uključuje i teretne, privatne i druge opažene letove.
`first_seen`/`last_seen` nisu planirana vremena niti pouzdana vremena izlaska/ulaska na stajanku.

Rang aerodroma određuje ukupan broj dolazaka i odlazaka u cijelom prihvaćenom
mjesečnom uzorku. Zatim zadržavamo samo veze između prvih `number_of_airports` čvorova.


In [3]:
airports = load_airports(data_paths.airports)
route_data = load_routes(data_paths.flights, airports; year, month)
network = build_network(route_data.routes, airports; n=number_of_airports)
display(result_table([
    (mjera="Godina", vrijednost=year),
    (mjera="Mjesec", vrijednost=month),
    (mjera="Redaka u mjesecu", vrijednost=route_data.selected),
    (mjera="Izvan razdoblja ili bez datuma", vrijednost=route_data.outside_period),
    (mjera="Nevaljani podaci ili izvan europskog skupa", vrijednost=route_data.invalid),
    (mjera="Prihvaćeni letovi prije izbora čvorova", vrijednost=route_data.accepted),
    (mjera="Aerodroma u modelu", vrijednost=length(network.codes)),
    (mjera="Letova unutar odabrane mreže", vrijednost=Int(sum(network.counts))),
]))


mjera,vrijednost
Godina,2025
Mjesec,1
Redaka u mjesecu,1036088
Izvan razdoblja ili bez datuma,0
Nevaljani podaci ili izvan europskog skupa,722313
Prihvaćeni letovi prije izbora čvorova,313775
Aerodroma u modelu,50
Letova unutar odabrane mreže,157446


## 3. Od mreže do stabilnog sustava
Neka je $C_{ij}$ broj letova iz aerodroma $i$ u aerodrom $j$. Definiramo

$$\widetilde W=(C+C^\mathsf T)/2,\quad
s=\max_i\sum_j\widetilde W_{ij},\quad W=\widetilde W/s,$$
$$D=\operatorname{diag}(W\mathbf1),\quad L=D-W,\quad A=-\alpha L-\gamma I.$$

Simetrizacija zanemaruje smjer letova. Jedinstveni faktor $s$ zadržava relativne
težine veza i daje usporedivu skalu za $\alpha$ i $\gamma$ unutar ovog eksperimenta.
Za simetričnu nenegativnu mrežu $L\succeq0$, pa vrijedi
$\lambda_i(A)=-\alpha\lambda_i(L)-\gamma<0$.
Član $-\gamma I$ nužan je: sam $-L$ ima nultu svojstvenu vrijednost.

Vrijeme je u proizvoljnim jedinicama modela. Parametri nisu procijenjeni iz vremena
letova ili kašnjenja. Izdvajanje podmreže zanemaruje sve veze s ostalim aerodromima.


In [4]:
A = system_matrix(network.L; alpha, gamma)
source_airports = network.codes[1:2]
B = disturbance_matrix(network.codes, source_airports; sigma, background=background_noise)
display((sources=source_airports, size_A=size(A), size_B=size(B),
         largest_eigenvalue=eigmax(Symmetric(A)),
         weight_scale=network.weight_scale))


(sources = ["EHAM", "EGLL"], size_A = (50, 50), size_B = (50, 52), largest_eigenvalue = -0.20000000000000012, weight_scale = 7995.0)

## 4. Stacionarna kovarijanca i Gramian
Promatramo linearni stohastički sustav
$$dx=A x\,dt+B\,d\mathcal W_t.$$
$x_i$ je apstraktno odstupanje od ravnoteže u aerodromu $i$, a $\mathcal W_t$
sadrži neovisne standardne Wienerove procese. Za stabilan $A$ kovarijanca zadovoljava
$$AX+XA^\mathsf T+BB^\mathsf T=0.$$
To je ujedno Gramian upravljivosti determinističkog sustava $\dot x=Ax+Bu$.
[Referenca za Gramian i Lyapunovljevu jednadžbu](https://www.mathworks.com/help/control/ref/statespacemodel.gram.html).

Svaki odabrani izvor daje stupac $\sigma e_i$ u $B$. Dodajemo i slabu nezavisnu
pozadinsku pobudu $\eta I$, s `η = background_noise`, tako da
$$BB^\mathsf T=\sigma^2\sum_{i\in S}e_ie_i^\mathsf T+\eta^2 I.$$
To izbjegava singularnu kovarijancu kod nedosegljivih modova. Pozadinska pobuda
jednaka je u svim usporedbama; može se isključiti postavljanjem vrijednosti na nulu.
Tada singularnost ili loša uvjetovanost mogu otežati faktorizaciju.

$X_{ii}$ mjeri modeliranu varijancu, $X_{ij}$ kovarijancu, a $\operatorname{tr}(X)$
ukupnu varijabilnost mreže. Te vrijednosti nemaju jedinicu minuta stvarnog kašnjenja.


In [5]:
U_real = lyapunov_factor(A, B)
U_complex = lyapunov_factor(A, B; arithmetic=:complex)
U_reference = MatrixEquations.plyapc(A, B)
X = Matrix(U_real * U_real')
X_reference = U_reference * U_reference'
solver_results = [
    (solver=name, residual=relative_residual(A, B, U),
     relative_error=norm(U * U' - X_reference) / norm(X_reference))
    for (name, U) in [("Sorensen–Zhou real", U_real),
                      ("Sorensen–Zhou complex", U_complex),
                      ("MatrixEquations.plyapc", U_reference)]
]
display(result_table(solver_results))
@assert maximum(row.residual for row in solver_results) < 1e-10
@assert maximum(row.relative_error for row in solver_results) < 1e-8


solver,residual,relative_error
Sorensen–Zhou real,9.446023706509116e-16,1.173950191847992e-14
Sorensen–Zhou complex,1.1248035307020093e-15,1.2091783779062205e-14
MatrixEquations.plyapc,2.6108541737445206e-15,0.0


## 5. Gdje su poremećaji najizraženiji?
Karta prikazuje geografski raspored bez kartografske podloge; debljina veze ovisi
o broju letova. Tamniji i veći čvor znači veću varijancu. Prelaskom miša preko
čvora dobivaju se naziv i vrijednost. Skala boje relativna je tom prikazu.
Toplinska karta pokazuje apsolutne kovarijance za prvih 20 aerodroma po prometu,
dok tablica rangira sve čvorove prema varijanci. Grafike su SVG, bez dodatnih servisa.


In [6]:
display(airport_map(network, diag(X)))
display(covariance_heatmap(network.codes, X))
ranking = sortperm(diag(X); rev=true)
display(result_table([
    (airport=network.codes[i], name=network.airports[i].name,
     monthly_traffic=network.traffic[i], variance=round(X[i, i]; sigdigits=5))
    for i in ranking[1:min(10, length(ranking))]
]))


HTML{String}("<svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 920 500' style='max-width:100%;background:#fff;color:#111;font-family:sans-serif'><text x='20' y='25'>Mreža aerodroma: tamnije / veće = veća modelirana varijanca</text><line x1='340.9883633846955' y1='192.14002258617109' x2='227.87151418599362' y2='205.9623951290621' stroke='#94a3b8' stroke-opacity='0.25' stroke-width='0.46923076923076923'/><line x1='340.9883633846955' y1='192.14002258617109' x2='423.15411279628404' y2='229.78529304979688' stroke='#94a3b8' stroke-opacity='0.25' stroke-width='0.40637898686679175'/><line x1='227.87151418599362' y1='205.9623951290621' x2='423.15411279628404' y2='229.78529304979688' stroke='#94a3b8' stroke-opacity='0.25' stroke-width='0.4596622889305816'/><line x1='340.9883633846955' y1='192.14002258617109' x2='293.1376403372806' y2='246.5754273168306' stroke='#94a3b8' stroke-opacity='0.25' stroke-width='0.42326454033771105'/><line x1='227.87151418599362' y1='205.9623951290621' x2='293.1376403372806' y2='246.5754273168306' stroke='#94a3b8' stroke-opacity='0.25' stroke-width='0.41294559099437145'/><line x1='423.15411279628404' y1='229.78529304979688' x2='293.1376403372806' y2='246.5754273168306' stroke='#94a3b8' stroke-opacity='0.25' stroke-width='0.3926829268292683'/><line x1='340.9883633846955' y1='192.14002258617109' x2='160.47648339803874' y2='387.059672136307' stroke='#94a3b8' stroke-opacity='0.25' stroke-width='0.3971857410881801'/><line x1='227.87151418599362' y1='205.9623951290621' x2='160.47648339803874' y2='387.059672136307' stroke='#94a3b8' stroke-opacity='0.25' stroke-width='0.44146341463414634'/><line x1='423.15411279628404' y1='229.78529304979688' x2='160.47648339803874' y2='387.059672136307' stroke='#94a3b8' stroke-opacity='0.25' stroke-width='0.3778611632270169'/><line x1='293.1376403372806' y1='246.5754273168306' x2='160.47648339803874' y2='387.059672136307' stroke='#94a3b8' stroke-opacity='0.25' stroke-width='0.3844277673545966'/><line x1='340.9883633846955' y1='192.14002258617109' x2='282.8376994055359' y2='373.8008498176977' stroke='#94a3b8' stroke-opacity='0.25' stroke-width='0.43358348968105065'/><line x1='227.87151418599362' y1='205.9623951290621' x2='282.8376994055359' y2='373.8008498176977' stroke='#94a3b8' stroke-opacity='0.25' stroke-width='0.40093808630393996'/><line x1='423.15411279628404' y1='229.78529304979688' x2='282.8376994055359' y2='373.8008498176977' stroke='#94a3b8' stroke-opacity='0.25' stroke-width='0.3439024390243902'/><line x1='293.1376403372806' y1='246.5754273168306' x2='282.8376994055359' y2='373.8008498176977' stroke='#94a3b8' stroke-opacity='0.25' stroke-width='0.39643527204502815'/><line x1='160.47648339803874' y1='387.059672136307' x2='282.8376994055359' y2='373.8008498176977' stroke='#94a3b8' stroke-opacity='0.25' stroke-width='0.47560975609756095'/><line x1='340.9883633846955' y1='192.14002258617109' x2='493.0482473143063' y2='257.3838124995003' stroke='#94a3b8' stroke-opacity='0.25' stroke-width='0.41031894934333957'/><line x1='227.87151418599362' y1='205.9623951290621' x2='493.0482473143063' y2='257.3838124995003' stroke='#94a3b8' stroke-opacity='0.25' stroke-width='0.4343339587242026'/><line x1='423.15411279628404' y1='229.78529304979688' x2='493.0482473143063' y2='257.3838124995003' stroke='#94a3b8' stroke-opacity='0.25' stroke-width='0.41819887429643526'/><line x1='293.1376403372806' y1='246.5754273168306' x2='493.0482473143063' y2='257.3838124995003' stroke='#94a3b8' stroke-opacity='0.25' stroke-width='0.38667917448405253'/><line x1='160.47648339803874' y1='387.059672136307' x2='493.0482473143063' y2='257.3838124995003' stroke='#94a3b8' stroke-opacity='0.25' stroke-width='0.375422138836773'/><line x1='282.8376994055359' y1='373.8008498176977' x2='493.0482473143063' y2='257.3838124995003' stroke='#94a3b8' stroke-opacity='0.25' stroke-width='0.35759849906191365'/><line x1='340.9883633846955' y1='192.14002258617109' x2='422.9312051297273' y2='272.16126739409924' stroke='#94a3b8' 

HTML{String}("<svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 560 530' style='max-width:650px;width:100%;background:#fff;font-family:sans-serif'><text x='15' y='18'>|Xᵢⱼ| za prvih 20 aerodroma; tamnije = veće</text><rect x='90' y='65' width='22' height='22' fill='rgb(30,120,165)'><title>EHAM / EHAM: 0.42692070500569806</title></rect><rect x='90' y='87' width='22' height='22' fill='rgb(222,237,249)'><title>EGLL / EHAM: 0.02754317669434832</title></rect><rect x='90' y='109' width='22' height='22' fill='rgb(229,242,253)'><title>EDDF / EHAM: 0.01170458612337481</title></rect><rect x='90' y='131' width='22' height='22' fill='rgb(229,241,252)'><title>LFPG / EHAM: 0.012372789171352936</title></rect><rect x='90' y='153' width='22' height='22' fill='rgb(230,242,253)'><title>LEMD / EHAM: 0.01049516732044687</title></rect><rect x='90' y='175' width='22' height='22' fill='rgb(229,241,252)'><title>LEBL / EHAM: 0.012524439923164238</title></rect><rect x='90' y='197' width='22' height='22' fill='rgb(229,242,253)'><title>EDDM / EHAM: 0.011852211306784993</title></rect><rect x='90' y='219' width='22' height='22' fill='rgb(229,241,252)'><title>LSZH / EHAM: 0.012691052091835476</title></rect><rect x='90' y='241' width='22' height='22' fill='rgb(228,241,252)'><title>EIDW / EHAM: 0.013925426969504617</title></rect><rect x='90' y='263' width='22' height='22' fill='rgb(229,241,252)'><title>EKCH / EHAM: 0.012708494693230803</title></rect><rect x='90' y='285' width='22' height='22' fill='rgb(231,243,253)'><title>LIRF / EHAM: 0.008433294443100196</title></rect><rect x='90' y='307' width='22' height='22' fill='rgb(231,243,253)'><title>LOWW / EHAM: 0.008221120303039554</title></rect><rect x='90' y='329' width='22' height='22' fill='rgb(231,242,253)'><title>LPPT / EHAM: 0.009298257697353502</title></rect><rect x='90' y='351' width='22' height='22' fill='rgb(232,243,254)'><title>EGKK / EHAM: 0.005953954735015628</title></rect><rect x='90' y='373' width='22' height='22' fill='rgb(230,242,253)'><title>LSGG / EHAM: 0.009994191152582639</title></rect><rect x='90' y='395' width='22' height='22' fill='rgb(231,242,253)'><title>EPWA / EHAM: 0.009353752562811183</title></rect><rect x='90' y='417' width='22' height='22' fill='rgb(230,242,253)'><title>EDDB / EHAM: 0.009767188916187533</title></rect><rect x='90' y='439' width='22' height='22' fill='rgb(232,243,253)'><title>EBBR / EHAM: 0.007276145982059493</title></rect><rect x='90' y='461' width='22' height='22' fill='rgb(234,244,254)'><title>EGSS / EHAM: 0.0030978847319422277</title></rect><rect x='90' y='483' width='22' height='22' fill='rgb(231,243,253)'><title>ENGM / EHAM: 0.007926246752244754</title></rect><rect x='112' y='65' width='22' height='22' fill='rgb(222,237,249)'><title>EHAM / EGLL: 0.02754317669434832</title></rect><rect x='112' y='87' width='22' height='22' fill='rgb(30,120,165)'><title>EGLL / EGLL: 0.4266520595063506</title></rect><rect x='112' y='109' width='22' height='22' fill='rgb(228,241,252)'><title>EDDF / EGLL: 0.01509265020507431</title></rect><rect x='112' y='131' width='22' height='22' fill='rgb(229,241,252)'><title>LFPG / EGLL: 0.0119684791351239</title></rect><rect x='112' y='153' width='22' height='22' fill='rgb(229,241,252)'><title>LEMD / EGLL: 0.013183137308160668</title></rect><rect x='112' y='175' width='22' height='22' fill='rgb(230,242,253)'><title>LEBL / EGLL: 0.01078901893094731</title></rect><rect x='112' y='197' width='22' height='22' fill='rgb(228,241,252)'><title>EDDM / EGLL: 0.01365367182179391</title></rect><rect x='112' y='219' width='22' height='22' fill='rgb(228,241,252)'><title>LSZH / EGLL: 0.014539423073198198</title></rect><rect x='112' y='241' width='22' height='22' fill='rgb(227,240,252)'><title>EIDW / EGLL: 0.016344151661647936</title></rect><rect x='112' y='263' width='22' height='22' fill='rgb(229,242,253)'><title>EKCH / EGLL: 0.011839274423998805</title></rect><rect x='112' y='285' width='22' height='22' fill='rgb(231,242,253)'><title>LIRF / EGLL: 0.008907

airport,name,monthly_traffic,variance
EHAM,Amsterdam Airport Schiphol,24422,0.42692
EGLL,London Heathrow Airport,20415,0.42665
EIDW,Dublin Airport,12075,0.003358
LSZH,Zürich Airport,12752,0.0028471
LSGG,Geneva International Airport,9966,0.0028438
EDDF,Frankfurt Main Airport,17098,0.0027233
EDDM,Munich Airport,15389,0.0025653
EKCH,Copenhagen Kastrup Airport,11977,0.0025031
EGPH,Edinburgh Airport,5918,0.0023729
EGCC,Manchester Airport,8096,0.0023333


## 6. Jednaka pobuda u prometnom i manje prometnom čvoru
Uspoređujemo prvi i posljednji aerodrom prema prometu **unutar odabranog skupa**.
Svaki scenarij ima jedan lokalni izvor iste jačine i jednaku pozadinsku pobudu.
Posebno izdvajamo varijancu izvan izvornog čvora: ona opisuje koliko se učinak
proširio na ostatak mreže. Velik lokalni poremećaj ne mora značiti velik utjecaj drugdje.


In [7]:
source_comparison = map([first(network.codes), last(network.codes)]) do source
    Bs = disturbance_matrix(network.codes, [source]; sigma, background=background_noise)
    Us = lyapunov_factor(A, Bs)
    Xs = Us * Us'
    index = findfirst(==(source), network.codes)
    (source=source, local_variance=Xs[index, index], total_variance=tr(Xs),
     variance_elsewhere=tr(Xs) - Xs[index, index], residual=relative_residual(A, Bs, Us))
end
display(result_table(source_comparison))


source,local_variance,total_variance,variance_elsewhere,residual
EHAM,0.42541637829334256,0.4626474984568721,0.03723112016352953,1.4580995643099333e-15
ENBR,1.6585592632111017,1.6782173304024106,0.019658067191308914,2.4648369252627744e-16


## 7. Utjecaj brzine oporavka
Mijenjamo samo $\gamma$, uz istu mrežu, normalizaciju, $\alpha$ i $B$.
Manji $\gamma$ znači sporiji oporavak. Za ovaj simetrični model
$$\operatorname{tr}(X)=\tfrac12\operatorname{tr}((\alpha L+\gamma I)^{-1}BB^\mathsf T),$$
pa ukupna varijanca pada s porastom $\gamma$. To ujedno daje nezavisnu provjeru
očekivanog ponašanja rezultata, uz provjeru reziduala.


In [8]:
recovery_results = map([0.02, 0.05, 0.1, 0.2, 0.5, 1.0]) do recovery_rate
    Ag = system_matrix(network.L; alpha, gamma=recovery_rate)
    Ug = lyapunov_factor(Ag, B)
    Xg = Ug * Ug'
    trace_check = tr((-Ag) \ (B * B')) / 2
    @assert isapprox(tr(Xg), trace_check; rtol=1e-8)
    (gamma=recovery_rate, total_variance=tr(Xg),
     max_variance=maximum(real, diag(Xg)), residual=relative_residual(Ag, B, Ug))
end
@assert all(diff(getproperty.(recovery_results, :total_variance)) .< 0)
display(result_table(recovery_results))
display(recovery_plot(recovery_results))


gamma,total_variance,max_variance,residual
0.02,1.984081234331855,0.5262282514448258,1.2392196517349303e-15
0.05,1.3508731546283965,0.49816522867369817,2.217355237453231e-15
0.1,1.1030102387747145,0.4699538151323447,1.200494433819213e-15
0.2,0.9229209745870968,0.42692070500569806,9.446023706509116e-16
0.5,0.6940527850112863,0.33797079351071296,7.846353996886615e-16
1.0,0.5094318698696447,0.25192194627518677,7.324741244340379e-16


HTML{String}("<svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 740 390' style='max-width:100%;background:#fff;font-family:sans-serif'><text x='30' y='25'>Ukupna stacionarna varijanca tr(X) i brzina oporavka γ</text><path d='M70 50 V320 H700' fill='none' stroke='#475569'/><text x='60' y='324' text-anchor='end'>0</text><text x='60' y='64' text-anchor='end' font-size='11'>1.98</text><polyline points='70.0,60.0 212.87680457596448,142.97750005096952 320.95893485865486,175.45820548118806 429.0410651413452,199.05764681381496 571.9178697173097,229.04922591856337 680.0,253.24250848493546' fill='none' stroke='#155e75' stroke-width='2'/><circle cx='70.0' cy='60.0' r='4' fill='#155e75'><title>γ=0.02, tr(X)=1.984081234331855</title></circle><text x='70.0' y='340' text-anchor='middle' font-size='11'>0.02</text><circle cx='212.87680457596448' cy='142.97750005096952' r='4' fill='#155e75'><title>γ=0.05, tr(X)=1.3508731546283965</title></circle><text x='212.87680457596448' y='340' text-anchor='middle' font-size='11'>0.05</text><circle cx='320.95893485865486' cy='175.45820548118806' r='4' fill='#155e75'><title>γ=0.1, tr(X)=1.1030102387747145</title></circle><text x='320.95893485865486' y='340' text-anchor='middle' font-size='11'>0.1</text><circle cx='429.0410651413452' cy='199.05764681381496' r='4' fill='#155e75'><title>γ=0.2, tr(X)=0.9229209745870968</title></circle><text x='429.0410651413452' y='340' text-anchor='middle' font-size='11'>0.2</text><circle cx='571.9178697173097' cy='229.04922591856337' r='4' fill='#155e75'><title>γ=0.5, tr(X)=0.6940527850112863</title></circle><text x='571.9178697173097' y='340' text-anchor='middle' font-size='11'>0.5</text><circle cx='680.0' cy='253.24250848493546' r='4' fill='#155e75'><title>γ=1.0, tr(X)=0.5094318698696447</title></circle><text x='680.0' y='340' text-anchor='middle' font-size='11'>1.0</text><text x='300' y='375' font-size='13'>γ (logaritamska os)</text></svg>")

## 8. Vrijeme izvođenja
Pokretanje je opcionalno. BenchmarkTools zagrijava funkcije i mjeri ponovljene
pozive na istim matricama; prikazujemo medijan. Učitavanje podataka i crtanje nisu
uključeni. Sve tri metode računaju faktor, a Schurova dekompozicija ulazi u mjerenje.
Simetrični $A$ ima realan spektar, pa ovaj primjer ne ispituje realne Schurove
blokove veličine 2. Takvi testovi ostaju u izvornoj bilježnici.


In [9]:
run_timing = false
if run_timing
    timings = [
        (solver="Sorensen–Zhou real", trial=@benchmark(lyapunov_factor($A, $B), samples=10, evals=1)),
        (solver="Sorensen–Zhou complex", trial=@benchmark(lyapunov_factor($A, $B; arithmetic=:complex), samples=10, evals=1)),
        (solver="MatrixEquations.plyapc", trial=@benchmark(MatrixEquations.plyapc($A, $B), samples=10, evals=1)),
    ]
    display(result_table([
        (solver=row.solver, median_ms=median(row.trial).time / 1e6,
         memory_bytes=median(row.trial).memory) for row in timings
    ]))
end


## Ograničenja i mogući nastavci
- Mreža prikazuje broj letova, bez kapaciteta zrakoplova, presjedanja i rasporeda.
- ADS-B pokrivenost i nepotpune oznake aerodroma utječu na rang i težine veza;
  prihvaćeni uzorak nije potpun popis europskog prometa ni komercijalnih linija.
- OPDI v0.0.2 i mjesec su zadani, a OurAirports snapshot je zaključan; lokalne
  datoteke čuvajte za ponavljanje jer se upstream OPDI datoteke mogu revidirati.
- Simetrizacija i difuzija su pretpostavke modela, a ne empirijski utvrđena dinamika.
- Pri usporedbi mjeseci normalizacija se ponovno računa; za čistu usporedbu
  sezonskih promjena trebalo bi zadržati iste čvorove i isti faktor normalizacije.
- Pozadinski šum osigurava pozitivno definitno rješenje, ali mijenja model.
- Nakon promjene parametara pokrenite sve ćelije od početka da prikazi budu usklađeni.

Daljnji korak bio bi procijeniti dinamiku iz vremenskih nizova kašnjenja i validirati
model na izdvojenim podacima. Diskretni model zahtijeva diskretnu Lyapunovljevu jednadžbu,
koju ovdje implementirani kontinuirani solver ne rješava izravno.
